In [24]:
import gensim
from gensim.models import Word2Vec
import gensim.downloader as api
import pandas as pd
from gensim.utils import simple_preprocess
from nltk import corpus, sent_tokenize

In [25]:
wv=api.load("word2vec-google-news-300")

In [26]:
messages=pd.read_csv("/home/sidd/Desktop/Spam vs Ham/spam.csv",encoding="latin")

In [27]:
messages

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [28]:
messages.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'],inplace=True)

In [29]:
messages

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [30]:
messages.rename(columns={
    "v1": "label",
    "v2": "text"},
            inplace=True)

In [31]:
!pip install nltk

In [32]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [33]:
import re

corpus=[]

for i in range(len(messages)):
    review=re.sub("[^a-zA-Z]"," ",messages['text'][i])
    review=review.lower()
    review=review.split()

    review=[lemmatizer.lemmatize(word) for word in review]
    review=" ".join(review)
    corpus.append(review)


In [34]:
corpus

['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
 'ok lar joking wif u oni',
 'free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry question std txt rate t c s apply over s',
 'u dun say so early hor u c already then say',
 'nah i don t think he go to usf he life around here though',
 'freemsg hey there darling it s been week s now and no word back i d like some fun you up for it still tb ok xxx std chgs to send to rcv',
 'even my brother is not like to speak with me they treat me like aid patent',
 'a per your request melle melle oru minnaminunginte nurungu vettam ha been set a your callertune for all caller press to copy your friend callertune',
 'winner a a valued network customer you have been selected to receivea prize reward to claim call claim code kl valid hour only',
 'had your mobile month or more u r entitled to update to the latest colour mobile with camera for free call the mobile up

In [35]:
words=[]

for sent in corpus:
    sent_token=sent_tokenize(sent)
    for word in sent_token:
        words.append(simple_preprocess(sent))    #it's basically a quick text-cleaning + tokenization function

In [36]:
model=gensim.models.Word2Vec(words,vector_size=100)

In [37]:
model.wv.index_to_key   #To check all the vocalbulary....

['you',
 'to',
 'the',
 'and',
 'it',
 'in',
 'is',
 'me',
 'my',
 'for',
 'your',
 'call',
 'of',
 'that',
 'have',
 'on',
 'now',
 'are',
 'can',
 'so',
 'but',
 'not',
 'or',
 'we',
 'do',
 'get',
 'at',
 'be',
 'if',
 'ur',
 'will',
 'with',
 'no',
 'just',
 'this',
 'gt',
 'lt',
 'how',
 'go',
 'up',
 'when',
 'ok',
 'day',
 'what',
 'free',
 'from',
 'out',
 'all',
 'know',
 'll',
 'come',
 'like',
 'time',
 'good',
 'am',
 'then',
 'got',
 'wa',
 'there',
 'he',
 'text',
 'only',
 'love',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'by',
 'going',
 'home',
 'don',
 'stop',
 'she',
 'about',
 'lor',
 'sorry',
 'see',
 'mobile',
 'still',
 'take',
 'back',
 'da',
 'reply',
 'our',
 'think',
 'tell',
 'dont',
 'week',
 'phone',
 'hi',
 'new',
 'later',
 'they',
 'any',
 'pls',
 'her',
 'please',
 'ha',
 'co',
 'msg',
 'did',
 'been',
 'min',
 'an',
 'some',
 'dear',
 'make',
 'here',
 'night',
 'who',
 'message',
 'well',
 'say',
 'where',
 're',
 'thing',
 'much',
 'clai

In [38]:
print(model.corpus_count)

5569


In [39]:
print(model.epochs)

5


In [40]:
model.wv["good"].shape

(100,)

# AVGWORD2VEC

In [41]:
import numpy as np


def avg_word2_vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [42]:
!pip install tqdm

In [43]:
from tqdm import tqdm

X=[]

for i in tqdm(range(len(words))):
    X.append(avg_word2_vec(words[i]))

100%|██████████| 5569/5569 [00:00<00:00, 15340.49it/s]


In [44]:
X

[array([-0.20176095,  0.21329832,  0.16437182,  0.108767  ,  0.13365953,
        -0.38778967,  0.23698078,  0.5713806 , -0.32191938, -0.11071613,
        -0.16108146, -0.36559093, -0.09212974,  0.09813336,  0.12791848,
        -0.15152997,  0.11456714, -0.32148972,  0.02923927, -0.5154097 ,
         0.19514221,  0.28401154,  0.10611294, -0.21240108, -0.05402562,
         0.07957723, -0.20624718, -0.17437014, -0.25853017,  0.0639642 ,
         0.20272703,  0.0190542 ,  0.14996734, -0.18061934, -0.05466012,
         0.3619035 , -0.05422745, -0.1885381 , -0.126298  , -0.43777338,
         0.08212686, -0.23389068, -0.22549348, -0.04255731,  0.24974194,
        -0.03821112, -0.12519836, -0.11486909,  0.1805741 ,  0.16147766,
         0.1224663 , -0.24876681, -0.12681964, -0.00617589, -0.14807658,
         0.11420312,  0.10793078,  0.01325799, -0.37400302,  0.14710662,
        -0.08313176,  0.06657152,  0.0390676 , -0.13016911, -0.35036677,
         0.3032072 ,  0.15888377,  0.2683979 , -0.3

In [45]:
len(X)

5569

In [46]:
X_new=np.array(X)   #Independent feature

In [47]:
X_new.shape  #we lost 3 records hawww

(5569, 100)

In [52]:
y=messages[list(map(lambda x: len(x)>0 , corpus))]
y=pd.get_dummies(y['label'])
y=y.iloc[:,0].values

In [53]:
y.shape

(5569,)

In [55]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test= train_test_split(X_new,y,test_size=0.2,random_state=42)

In [57]:
from sklearn.ensemble import RandomForestClassifier

model=RandomForestClassifier(n_jobs=-1,n_estimators=100)

In [58]:
model.fit(X_train,y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether boo

In [59]:
y_pred=model.predict(X_test)

In [60]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.9640933572710951
Precision: 0.9760166840458812
Recall   : 0.9821615949632738
F1 Score : 0.9790794979079498

Confusion Matrix:
[[138  23]
 [ 17 936]]

Classification Report:
              precision    recall  f1-score   support

       False       0.89      0.86      0.87       161
        True       0.98      0.98      0.98       953

    accuracy                           0.96      1114
   macro avg       0.93      0.92      0.93      1114
weighted avg       0.96      0.96      0.96      1114

